In [1]:
import pandas as pd
import os
import numpy as np
import rdkit
from pyopsin import PyOpsin
from rdkit import Chem,DataStructs
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, MolSurf
from rdkit.Chem import rdFingerprintGenerator
from propy.PyPro import GetProDes
from Bio.SeqUtils.ProtParam import ProteinAnalysis
pd.set_option('display.max_colwidth', None)

C:\Users\blons\code\trial\envs\chem_env\Lib\site-packages\pyopsin\pyopsin.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


# Drug properties

In [15]:
def drug_properties(smiles):

    smiles = str(smiles)
        
    if "|" in smiles:
        smiles = smiles.split("|")[0].strip()
    
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return False
    #Bond features

    single = 0
    double = 0
    triple = 0
    aromatic = 0
    conjugated = 0
    total_bonds = mol.GetNumBonds()
    ringBonds = 0
    
    for bond in mol.GetBonds():
        if bond.IsInRing():
            ringBonds +=1
        btype = bond.GetBondType()
        if btype == Chem.rdchem.BondType.SINGLE:
            single += 1
        elif btype == Chem.rdchem.BondType.DOUBLE:
            double += 1
        elif btype == Chem.rdchem.BondType.TRIPLE:
            triple += 1
        elif btype == Chem.rdchem.BondType.AROMATIC:
            aromatic += 1
        
        if bond.GetIsConjugated():
            conjugated += 1

    
   #Hybridization
    
    SP3 = 0
    SP2 = 0
    SP = 0
    
    for i in mol.GetAtoms():
        hybridization = i.GetHybridization()
        if hybridization == Chem.rdchem.HybridizationType.SP3:
            SP3 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP2:
            SP2 +=1
        elif hybridization == Chem.rdchem.HybridizationType.SP:
            SP +=1


    #formalCharge
    
    overall_charge = 0
    for i in mol.GetAtoms():
        overall_charge += i.GetFormalCharge()


    #Aromatic Cycles
    heteroAromatic = Lipinski.NumAromaticHeterocycles(mol)
    carboAromatic = Lipinski.NumAromaticCarbocycles(mol)

    
    #TPSA
    
    Tpsa = MolSurf.TPSA(mol)


    #Hydrogen Bonds
    
    Hdonors = Lipinski.NumHDonors(mol)
    Hacceptors = Lipinski.NumHAcceptors(mol)

    
    #logP
    
    logp = Crippen.MolLogP(mol)

    
    #Mr
    
    weight = Descriptors.ExactMolWt(mol)

    
    #Morgan fingerprint
    morgan_fp_gen = rdFingerprintGenerator.GetMorganGenerator(includeChirality=True, radius=2, fpSize=2048,countSimulation=False)
    
    fp = morgan_fp_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    output = {"Molecular weight": weight,
             "LogP": logp,
             "Hydrogen bond Donors": Hdonors,
             "Hydrogen bond Acceptors": Hacceptors,
             "TPSA": Tpsa,
             "Aromatic Heterocycles": heteroAromatic,
             "Aromatic Carbocycles": carboAromatic,
             "Overall charge": overall_charge,
             'SP3' : SP3,
             'SP2' : SP2,
             'SP' : SP,
             "Single bonds": single,
             "Double bonds": double,
             "Triple bonds": triple,
             "Aromatic bonds": aromatic,
             "Rotatable bonds": Lipinski.NumRotatableBonds(mol),
             "Conjugated bonds": conjugated,
             "Conjugation ratio": conjugated / total_bonds if total_bonds else 0,
             "Bonds in ring": ringBonds
            }
    for i, bit in enumerate(arr):
        output[f"drug_FP_{i}"] = int(bit)
    return output

# Protein properties

In [19]:
def protein_properties(sequence):

    sequence = sequence.upper()
    
    allowed = set("ARNDCEQGHILKMFPSTWYV")
     
    if set(sequence) - allowed:
        return False
 
    des = GetProDes(sequence)
    analysis = ProteinAnalysis(sequence)
  
    protein_features = {}

    
    AAC = des.GetAAComp()
    DPC = des.GetDPComp()
    CTD = des.GetCTD()
    PAAC = des.GetPAAC(lamda=10, weight=0.05)
    APAAC = des.GetAPAAC(lamda=10, weight=0.05)
    moreau_broto = des.GetMoreauBrotoAuto()
    moran = des.GetMoranAuto()
    geary = des.GetGearyAuto()
    Mr = analysis.molecular_weight()
    aromaticity = analysis.aromaticity()
    IP = analysis.isoelectric_point()
    II = analysis.instability_index()
    GRAVY = analysis.gravy()

    protein_features.update(AAC)
    protein_features.update(DPC)
    protein_features.update(CTD)
    protein_features.update(PAAC)
    protein_features.update(APAAC)
    protein_features.update(moreau_broto)
    protein_features.update(moran)
    protein_features.update(geary)
    protein_features.update({"Mr" : Mr,
                            "Aromaticity" : aromaticity,
                            "Isoelectric point" : IP,
                            "Instability index" : II,
                            "GRAVY" : GRAVY})
 
    return protein_features

# Applying the properties

In [23]:
trial_num = int(input("Enter trial number: "))
chunk_num = int(input("Enter chunk number: ")) 

all_properties = []
    
chunk = pd.read_csv(f"chunks/chunk_{chunk_num}_{trial_num}.csv")
    
print(chunk.iloc[:,4].nunique())
print(chunk.iloc[:,2].nunique())
print(len(chunk))

protein_cache = {}
drug_cache = {}
    
unique_sequences = chunk.iloc[:, 4].dropna().unique()
unique_SMILES = chunk.iloc[:, 2].dropna().unique()

seq_num = 0

for seq in unique_sequences:
    
    protein_cache[seq] = protein_properties(seq)
    
    if protein_cache[seq] == False:
        print('Incorrect sequence!\n', seq)
        
    seq_num += 1
    print('Sequences processed: ', seq_num)

smiles_num = 0

for smiles in unique_SMILES:
    
    drug_cache[smiles] = drug_properties(smiles)
    
    if drug_cache[smiles] == False:
        print('Incorrect SMILES!\n', smiles)
        
    smiles_num += 1
    print('SMILES processed: ', smiles_num)
    
    
# for row in chunk.itertuples(index=False):
        
#     smiles = row[2]
#     sequence = row[4]
#     ki = row[3]

#     if properties == False:
#         print('---------------------------------\n ERROR \n -------------------------------', smiles,'\n', sequence)
#         continue
        
#     properties["Ki"] = ki

#     all_properties.append(properties)

# df = df.DataFrame(all_properties)
# df.to_csv(f"/Dataframes/chunk_{i}_{trial_num}_dataframe.csv")
    
# print(f"Chunk {i} done!")

Enter trial number:  1
Enter chunk number:  0


523
9394
19309
Sequences processed:  1
Sequences processed:  2
Sequences processed:  3
Sequences processed:  4
Sequences processed:  5
Sequences processed:  6
Sequences processed:  7
Sequences processed:  8
Sequences processed:  9
Sequences processed:  10
Sequences processed:  11
Sequences processed:  12
Sequences processed:  13
Sequences processed:  14
Sequences processed:  15
Sequences processed:  16
Sequences processed:  17
Sequences processed:  18
Sequences processed:  19
Sequences processed:  20
Sequences processed:  21
Sequences processed:  22
Sequences processed:  23
Sequences processed:  24
Sequences processed:  25
Sequences processed:  26
Sequences processed:  27
Sequences processed:  28
Sequences processed:  29
Sequences processed:  30
Incorrect sequence!
 PQITLWKRPLVTIRIGGQLKEALLDTGADDTVIEEMNLPGKWKPKMIGGIGGFIKVRQYDQIPVEIBGHKAIGTVLVGPTPVNIIGRNLLTQIGBTLNF
Sequences processed:  31
Sequences processed:  32
Sequences processed:  33
Sequences processed:  34
Sequences processed:  3